# Plot From A Batch Case


This notebook loads a completed case from a batch run using only the batch run name and the case key.


It reads the batch summary, resolves the corresponding result directory, loads the original and permuted matrices, and then displays the plots you request for that case.

In [ ]:
import gc
import matplotlib.pyplot as plt
from pathlib import Path
import traceback
import importlib
import notebook_functions

importlib.reload(notebook_functions)
from notebook_functions import *

ROOT_OUTPUT = Path("results_images")
ROOT_OUTPUT.mkdir(exist_ok=True, parents=True)

BATCH_RUN_NAME = "batch_20260521_151902"
completed_rows = get_completed_rows(BATCH_RUN_NAME)
DEMONSTRATORS = ["demonstrator_02"]

print(f"Processing demonstrators: {DEMONSTRATORS}")

METHODS = ["qaoa_hardware", "qaoa_emulated", "qa_hardware", "qa_emulated", "metis"]

PLOT_SPEC = [
    ("k_original_matrix_partitioned", True),
    ("k_permuted_matrix_partitioned", False),
    ("k_original_graph_partitioned", True),
    ("k_coarsened_graph_partitioned", False),
    ("k_uncoarsened_graph_partitioned", False),
    ("m_original_matrix_partitioned", True),
    ("m_permuted_matrix_partitioned", False),
    ("m_original_graph_partitioned", True),
    ("m_coarsened_graph_partitioned", False),
    ("m_uncoarsened_graph_partitioned", False),
]

def find_row_for(case_key):
    for r in completed_rows:
        if r.get("case_key") == case_key:
            return r
    return None

created_files = []

for dem in DEMONSTRATORS:
    dem_short = "dem" + "".join([c for c in dem if c.isdigit()])
    for method in METHODS:
        sizes = sorted({r["case_key"].split("__")[-1] for r in completed_rows if r["case_key"].startswith(f"{dem}__{method}__")})
        if not sizes:
            print(f"No completed cases for {dem} / {method}. Skipping.")
            continue
        for size in sizes:
            case_key = f"{dem}__{method}__{size}"
            row = find_row_for(case_key)
            if row is None:
                print(f"Case {case_key} not found in completed_rows. Skipping.")
                continue
            try:
                case = load_case_from_summary(row)
            except Exception as e:
                print(f"Failed to load case {case_key}: {e}")
                traceback.print_exc()
                continue
            out_dir = ROOT_OUTPUT / dem_short / method / f"coarsen_{size}"
            out_dir.mkdir(parents=True, exist_ok=True)
            for plot_name, show_y in PLOT_SPEC:
                out_file = out_dir / f"{plot_name}.png"
                try:
                    render_case_plot(case, method, plot_name, display_width=700, display_height=700, title_fontsize=16, label_fontsize=14, tick_labelsize=12, show_y_axis=show_y, save_path=str(out_file))
                    created_files.append(out_file)
                    print(f"Saved {out_file}")
                except Exception as e:
                    print(f"Failed to render {plot_name} for {case_key}: {e}")
                    traceback.print_exc()
                finally:
                    plt.close("all")

            # Curve comparisons for K and M
            if method != "metis":
                metis_key = f"{dem}__metis__{size}"
                metis_row = find_row_for(metis_key)
                if metis_row is None:
                    print(f"No metis counterpart for {case_key} (expected {metis_key}). Skipping curve.")
                else:
                    metis_case = None
                    try:
                        metis_case = load_case_from_summary(metis_row)
                        if case.get("run_metrics") is None or metis_case.get("run_metrics") is None:
                            print(f"Missing run_metrics for curve comparison {case_key} vs {metis_key}. Skipping curve.")
                        else:
                            # K curve comparison
                            out_curve_k = out_dir / f"k_curve_comparison_{method}_vs_metis.png"
                            render_k_curve_comparison(case, metis_case, title=f"{NAME_MAPPING.get(method, method)} {NAME_MAPPING.get(size,size)} vs Metis {NAME_MAPPING.get(size,size)}", backend_label=f"{NAME_MAPPING.get(method,method)} {NAME_MAPPING.get(size,size)}", metis_label=f"Metis {NAME_MAPPING.get(size,size)}", display_width=1200, display_height=700, title_fontsize=16, label_fontsize=14, tick_labelsize=12, save_path=str(out_curve_k))
                            created_files.append(out_curve_k)
                            print(f"Saved {out_curve_k}")
                            # M curve comparison
                            if hasattr(notebook_functions, "render_m_curve_comparison"):
                                out_curve_m = out_dir / f"m_curve_comparison_{method}_vs_metis.png"
                                notebook_functions.render_m_curve_comparison(case, metis_case, title=f"{NAME_MAPPING.get(method, method)} {NAME_MAPPING.get(size,size)} vs Metis {NAME_MAPPING.get(size,size)} (M)", backend_label=f"{NAME_MAPPING.get(method,method)} {NAME_MAPPING.get(size,size)}", metis_label=f"Metis {NAME_MAPPING.get(size,size)}", display_width=1200, display_height=700, title_fontsize=16, label_fontsize=14, tick_labelsize=12, save_path=str(out_curve_m))
                                created_files.append(out_curve_m)
                                print(f"Saved {out_curve_m}")
                    except Exception as e:
                        print(f"Failed curve comparison for {case_key} vs {metis_key}: {e}")
                        traceback.print_exc()
                    finally:
                        plt.close("all")
                        del metis_case
                        metis_case = None

            # Free case and force cleanup
            del case
            case = None
            plt.close("all")
            gc.collect()

    gc.collect()

print(f"Done. Created {len(created_files)} images.")

In [12]:
import importlib
import json
from pathlib import Path

import notebook_functions

importlib.reload(notebook_functions)
from notebook_functions import *

BATCH_RUN_NAME = "batch_20260521_151902"
DEMONSTRATOR = "demonstrator_01"

repo_root = Path(REPO_ROOT) if "REPO_ROOT" in globals() else Path.cwd()
summary_path = repo_root / "batch" / "runs" / BATCH_RUN_NAME / "batch_summary.json"
if not summary_path.exists():
    print(f"Batch summary not found: {summary_path}")
else:
    with summary_path.open() as fh:
        batch_summary = json.load(fh)

    rows = batch_summary.get("rows", [])
    demo_rows = [
        row
        for row in rows
        if row.get("status") == "completed"
        and row.get("case_key", "").startswith(f"{DEMONSTRATOR}__")
    ]

    if not demo_rows:
        print(f"No completed rows found for {DEMONSTRATOR} in {BATCH_RUN_NAME}")
    else:
        demo_suffix = "".join(ch for ch in DEMONSTRATOR if ch.isdigit())

        # K-matrix table
        k_lines = build_matrix_summary_lines("K", DEMONSTRATOR, demo_rows, demo_suffix=demo_suffix)
        print("\n".join(k_lines))

        # M-matrix table
        m_lines = build_matrix_summary_lines("M", DEMONSTRATOR, demo_rows, demo_suffix=demo_suffix)
        print("\n".join(m_lines))

\begin{table}[H]
\caption{K-matrix numerical summary for Demonstrator~01}
\scriptsize
\renewcommand{\arraystretch}{1.15}
\begin{tabularx}{\textwidth}{X l r r r r r r}
\toprule
\textbf{Method} & \textbf{Size} & \textbf{Bandwidth} & \textbf{Avg. Bandwidth} & \textbf{$\mathrm{frac}_{0.1\%}$} & \textbf{$\mathrm{frac}_{0.4\%}$} & \textbf{$\mathrm{frac}_{0.7\%}$} & \textbf{$\mathrm{frac}_{1.0\%}$}\\
\midrule
Original & N/A & 309635 & 2695.46 & 0.2926 & 0.6724 & 0.8515 & 0.9234\\
METIS & big & 251021 & 7413.11 & 0.3318 & 0.7056 & 0.8483 & 0.9019\\
METIS & medium & 310781 & 6404.33 & 0.3386 & 0.7071 & 0.8529 & 0.9087\\
METIS & small & 258956 & 4994.66 & 0.3848 & 0.7795 & 0.8917 & 0.9289\\
QA hardware & medium & 311540 & 6661.11 & 0.3404 & 0.7237 & 0.8633 & 0.9155\\
QA hardware & small & 306515 & 5680.84 & 0.3630 & 0.7500 & 0.8756 & 0.9201\\
QA emulated & big & 250511 & 3770.20 & 0.3070 & 0.6815 & 0.8484 & 0.9137\\
QA emulated & medium & 283400 & 5610.28 & 0.3341 & 0.6998 & 0.8482 & 0.9047\\
QA

In [13]:
# LaTeX tables for K and M matrix nB^2 metrics
import json
from pathlib import Path

BATCH_RUN_NAME = "batch_20260521_151902"
DEMONSTRATOR = "demonstrator_01"

repo_root = Path(REPO_ROOT) if "REPO_ROOT" in globals() else Path.cwd()
summary_path = repo_root / "batch" / "runs" / BATCH_RUN_NAME / "batch_summary.json"
if not summary_path.exists():
    print(f"Batch summary not found: {summary_path}")
else:
    with summary_path.open() as fh:
        batch_summary = json.load(fh)

    rows = batch_summary.get("rows", [])
    demo_rows = [
        row
        for row in rows
        if row.get("status") == "completed"
        and row.get("case_key", "").startswith(f"{DEMONSTRATOR}__")
    ]

    if not demo_rows:
        print(f"No completed rows found for {DEMONSTRATOR} in {BATCH_RUN_NAME}")
    else:
        method_order = ["metis", "qa_hardware", "qa_emulated", "qaoa_emulated", "qaoa_hardware"]
        size_order = ["big", "medium", "small"]
        row_by_case = {row["case_key"]: row for row in demo_rows}
        row_end = chr(92) * 2

        def pretty_method(method: str) -> str:
            return {
                "metis": "METIS",
                "qa_hardware": "QA hardware",
                "qa_emulated": "QA emulated",
                "qaoa_emulated": "QAOA emulated",
                "qaoa_hardware": "QAOA hardware",
            }.get(method, method)

        def get_n_from_matrix(row, prefix):
            try:
                result_dir = repo_root / row["result_dir"]
                if prefix == "K":
                    mtx_path = result_dir / "matrix_k_permuted.mtx"
                    if not mtx_path.exists():
                        mtx_path = result_dir.parent.parent / "data" / "matrices" / row["simulation_config"]["input_matrices"]["matrix_k_file_path"]
                else:
                    mtx_path = result_dir / "matrix_m_permuted.mtx"
                    if not mtx_path.exists():
                        mtx_path = result_dir.parent.parent / "data" / "matrices" / row["simulation_config"]["input_matrices"]["matrix_m_file_path"]
                with open(mtx_path, "r") as f:
                    for line in f:
                        if not line.startswith("%"):
                            shape = line.strip().split()
                            ncols = int(shape[1])
                            return ncols
            except Exception:
                return None
            return None

        def fmt_sci(value):
            if value is None:
                return "N/A"
            try:
                return f"{value:.2e}"
            except Exception:
                return "N/A"

        for prefix in ["K"]:
            table_rows = []
            for method in method_order:
                for size in size_order:
                    case_key = f"{DEMONSTRATOR}__{method}__{size}"
                    row = row_by_case.get(case_key)
                    if row is None:
                        continue
                    nnz_orig = row.get(f"{prefix}_nnz_original", None)
                    nnz_perm = row.get(f"{prefix}_nnz_permuted", None)
                    nnz_diff = None
                    if nnz_orig is not None and nnz_perm is not None:
                        nnz_diff = abs(nnz_orig - nnz_perm)
                    max_err = row.get(f"{prefix}_permutation_max_abs_err", None)
                    bw_orig = row.get(f"{prefix}_bandwidth_original", None)
                    bw_perm = row.get(f"{prefix}_bandwidth_permuted", None)
                    n = row.get("matrix_ncols", None)
                    if n is None:
                        n = get_n_from_matrix(row, prefix)
                    nB2_orig = None
                    nB2_perm = None
                    if n is not None and bw_orig is not None:
                        nB2_orig = n * (bw_orig ** 2)
                    if n is not None and bw_perm is not None:
                        nB2_perm = n * (bw_perm ** 2)
                    nB2_reduction = None
                    if nB2_orig is not None and nB2_perm is not None and nB2_orig != 0:
                        nB2_reduction = 100.0 * (nB2_orig - nB2_perm) / nB2_orig
                    table_rows.append(
                        {
                            "method": pretty_method(method),
                            "size": size,
                            "nnz_diff": str(nnz_diff) if nnz_diff is not None else "N/A",
                            "max_err": str(max_err) if max_err is not None else "N/A",
                            "nB2_orig": fmt_sci(nB2_orig),
                            "nB2_perm": fmt_sci(nB2_perm),
                            "nB2_reduction": f"{nB2_reduction:.1f}\%" if nB2_reduction is not None else "N/A",
                        }
                    )

            demo_suffix = "".join(ch for ch in DEMONSTRATOR if ch.isdigit())
            demo_display = f"Demonstrator~{demo_suffix}" if demo_suffix else DEMONSTRATOR
            demo_label = f"tab:results_demo{int(demo_suffix)}_{prefix.lower()}_nb2" if demo_suffix else f"tab:results_{prefix.lower()}_nb2"

            lines = [
                "\\begin{table}[H]",
                f"\\caption{{{prefix}-matrix $n B^2$ metrics for {demo_display}}}",
                "\\scriptsize",
                "\\renewcommand{\\arraystretch}{1.15}",
                "\\begin{tabularx}{\\textwidth}{X l r r r r r}",
                "\\toprule",
                "\\textbf{Method} & \\textbf{Size} & \\textbf{NNZ Diff} & \\textbf{Max Abs Err} & $n B^2_\mathrm{original}$ & $n B^2_\mathrm{permuted}$ & $n B^2$ \\% Reduction" + row_end,
                "\\midrule",
            ]
            for row in table_rows:
                lines.append(
                    f"{row['method']} & {row['size']} & {row['nnz_diff']} & {row['max_err']} & {row['nB2_orig']} & {row['nB2_perm']} & {row['nB2_reduction']}" + row_end
                )
            lines.extend([
                "\\bottomrule",
                "\\end{tabularx}",
                "\\centering",
                "Source: Author (2026)",
                f"\\label{{{demo_label}}}",
                "\\end{table}",
            ])
            print("\n".join(lines))

\begin{table}[H]
\caption{K-matrix $n B^2$ metrics for Demonstrator~01}
\scriptsize
\renewcommand{\arraystretch}{1.15}
\begin{tabularx}{\textwidth}{X l r r r r r}
\toprule
\textbf{Method} & \textbf{Size} & \textbf{NNZ Diff} & \textbf{Max Abs Err} & $n B^2_\mathrm{original}$ & $n B^2_\mathrm{permuted}$ & $n B^2$ \% Reduction\\
\midrule
METIS & big & 0 & 0.0 & 3.00e+16 & 1.97e+16 & 34.3\%\\
METIS & medium & 0 & 0.0 & 3.00e+16 & 3.02e+16 & -0.7\%\\
METIS & small & 0 & 0.0 & 3.00e+16 & 2.10e+16 & 30.1\%\\
QA hardware & medium & 0 & 0.0 & 3.00e+16 & 3.03e+16 & -1.2\%\\
QA hardware & small & 0 & 0.0 & 3.00e+16 & 2.94e+16 & 2.0\%\\
QA emulated & big & 0 & 0.0 & 3.00e+16 & 1.96e+16 & 34.5\%\\
QA emulated & medium & 0 & 0.0 & 3.00e+16 & 2.51e+16 & 16.2\%\\
QA emulated & small & 0 & 0.0 & 3.00e+16 & 1.09e+16 & 63.8\%\\
QAOA emulated & small & 0 & 0.0 & 3.00e+16 & 3.05e+16 & -1.7\%\\
\bottomrule
\end{tabularx}
\centering
Source: Author (2026)
\label{tab:results_demo1_k_nb2}
\end{table}
